# Notebook 01 — Descriptive Measurements (M1–M6)

**Part of:** Paper 3 reproducibility package — run this notebook once per model.

**What this does:** Runs six descriptive measurements (M1–M6) on a single transformer model,
producing per-layer summary CSVs used by notebook 03 (audit table) and 04 (aggregation).

**Inputs:**
- `../config.yaml` — hyperparameters, model IDs, band definitions
- `../data/prompts.json` — convergence prompts, M6 difficulty prompts

**Outputs (saved to current directory or configured OUTPUT_DIR):**
- `{MODEL_PRESET}_m1_gain_crossover.csv`
- `{MODEL_PRESET}_m2_logit_lens.csv`, `{MODEL_PRESET}_m2_logit_lens_summary.csv`
- `{MODEL_PRESET}_m3_similarity.csv`, `{MODEL_PRESET}_m3_similarity_summary.csv`
- `{MODEL_PRESET}_m4_ablation.csv`, `{MODEL_PRESET}_m4_ablation_summary.csv`
- `{MODEL_PRESET}_m5_alignment.csv`, `{MODEL_PRESET}_m5_alignment_summary.csv`
- `{MODEL_PRESET}_m6_per_prompt.csv`, `{MODEL_PRESET}_m6_per_prompt_convergence.csv`
- `{MODEL_PRESET}_convergence_crossovers.csv`, `{MODEL_PRESET}_confirmation_checks.csv`

**Hardware:** GPU recommended. GPT-2: ~8GB VRAM, Gemma-2-2B: ~16GB VRAM, Qwen2.5-1.5B: ~12GB VRAM.

**Runtime:** Full mode ~2–4 GPU hours per model. Set `fast_mode.enabled: true` in config.yaml for ~30 min.

**To switch models:** Change `MODEL_PRESET` in the config cell below.
For Gemma-2-2B, set `HF_TOKEN` environment variable before running.

In [ ]:
# ── Load shared config and prompts ────────────────────────────────────────────
# This cell replaces hardcoded constants in the original notebook.
# All magic numbers now live in ../config.yaml; all prompts in ../data/prompts.json.
import yaml, json, os

cfg   = yaml.safe_load(open('../config.yaml'))
pdata = json.load(open('../data/prompts.json'))

# ── Select model ───────────────────────────────────────────────────────────────
MODEL_PRESET = 'gpt2'   # change to: 'gemma2_2b' | 'qwen2_1_5b'
model_cfg    = cfg['models'][MODEL_PRESET]
exp_cfg      = cfg['experiment']
fast_cfg     = cfg['fast_mode']

FAST_MODE    = fast_cfg['enabled']
HF_ID        = model_cfg['hf_id']
N_LAYERS     = model_cfg['n_layers']

# ── HF token (required for Gemma; safe to leave unset for GPT-2 / Qwen) ──
HF_TOKEN = os.environ.get("HF_TOKEN", None)
if model_cfg.get("requires_hf_token") and not HF_TOKEN:
    raise EnvironmentError(
        "HF_TOKEN environment variable is not set.\n"
        "Gemma models require a Hugging Face token.\n"
        "Set it with: export HF_TOKEN=hf_..."
    )

# ── Prompts (from shared prompts.json) ────────────────────────────────────────────
CONVERGENCE_PROMPTS = pdata['convergence_prompts']
EASY_PROMPTS        = pdata['m6_difficulty']['easy']
MEDIUM_PROMPTS      = pdata['m6_difficulty']['medium']
HARD_PROMPTS        = pdata['m6_difficulty']['hard']

if FAST_MODE:
    n_conv = fast_cfg['convergence_prompts']
    n_diff = fast_cfg['m6_prompts_per_difficulty']
    CONVERGENCE_PROMPTS = CONVERGENCE_PROMPTS[:n_conv]
    EASY_PROMPTS        = EASY_PROMPTS[:n_diff]
    MEDIUM_PROMPTS      = MEDIUM_PROMPTS[:n_diff]
    HARD_PROMPTS        = HARD_PROMPTS[:n_diff]

# ── Experiment hyperparameters ────────────────────────────────────────────
M1_N_RAND       = fast_cfg['m1_n_rand'] if FAST_MODE else exp_cfg['m1_n_rand']
M1_N_PROMPTS    = exp_cfg['m1_n_prompts']
M4_N_PROMPTS    = exp_cfg['m4_n_prompts']
RIDGE_ALPHA     = exp_cfg['ridge_alpha']
PERTURB_SCALE   = exp_cfg['perturb_scale']

# ── Output directory ─────────────────────────────────────────────────────
OUTPUT_DIR = '.'   # outputs go to current working directory by default
PREFIX     = os.path.join(OUTPUT_DIR, MODEL_PRESET)

print(f'Model:       {HF_ID}  ({N_LAYERS} layers)')
print(f'Fast mode:   {FAST_MODE}')
print(f'Convergence: {len(CONVERGENCE_PROMPTS)} prompts')
print(f'M6 prompts:  {len(EASY_PROMPTS)+len(MEDIUM_PROMPTS)+len(HARD_PROMPTS)} total')
print(f'M1 n_rand:   {M1_N_RAND}')
print(f'Output prefix: {PREFIX}')
print(f'HF_TOKEN:    {"set" if HF_TOKEN else "not set (OK unless using Gemma)"}')

# GPT-2 Rerun Confirmation (Lean)

This notebook is a lean rerun to confirm the two-stage GPT-2 thesis quickly:
- latent structure appears early (casted lens)
- explicit token commitment appears late (raw lens + final-layer jumps)
- hourglass bottleneck pattern is present (ablation + update alignment)

Toggle `FAST_MODE` in the config cell.


## 0. Setup

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 0: INSTALL + IMPORTS
# ════════════════════════════════════════════════════════════════
!pip install -q nnsight transformers safetensors huggingface_hub
!pip install -q matplotlib scipy seaborn pandas tqdm scikit-learn

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from numpy.linalg import lstsq
from tqdm.auto import tqdm
import gc, os, warnings, time
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

torch.set_grad_enabled(False)
np.random.seed(42)

# Timing tracker
TIMING = {}
def tick(name): TIMING[name] = time.time()
def tock(name):
    if name in TIMING:
        elapsed = time.time() - TIMING[name]
        print(f'  {name}: {elapsed:.0f}s')
        TIMING[name] = elapsed
        return elapsed

# PREFIX and OUTPUT_DIR are already set by Cell 1 (config cell).
# They are NOT redefined here so that MODEL_PRESET drives the output filenames.
print(f'Output prefix: {PREFIX}')
print(f'Model: {HF_ID}  ({N_LAYERS} layers)')
print('Setup complete')


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 1: LOAD MODEL + INFRASTRUCTURE
# ════════════════════════════════════════════════════════════════
# MODEL_ID (= HF_ID from config) and N_LAYERS are set by the config cell above.

from nnsight import LanguageModel
from transformers import AutoTokenizer

print(f'Loading {HF_ID}...')
_hf_kwargs = {"trust_remote_code": True}
if HF_TOKEN:
    _hf_kwargs["token"] = HF_TOKEN

tokenizer = AutoTokenizer.from_pretrained(HF_ID, **_hf_kwargs)
model = LanguageModel(HF_ID, device_map='auto', **_hf_kwargs)
print('Model loaded.')

# ── Architecture-agnostic layer access ──────────────────────────────────────
# nnsight exposes the underlying HuggingFace model at model._model.
# Different model families use different attribute names for the transformer blocks.
def _get_layer_list(model_obj):
    """Return the list of transformer layer modules, regardless of architecture."""
    inner = model_obj._model
    for attr in ('model.layers', 'transformer.h', 'gpt_neox.layers', 'layers'):
        obj = inner
        for part in attr.split('.'):
            obj = getattr(obj, part, None)
            if obj is None: break
        if obj is not None and hasattr(obj, '__len__') and len(obj) > 0:
            print(f'  Layer path: {attr} ({len(obj)} layers)')
            return obj, attr
    raise RuntimeError('Cannot locate transformer layers. Add this model architecture to _get_layer_list.')

_layers_list, _layer_attr = _get_layer_list(model)
# Expose as a nnsight-traceable attribute string, e.g. "transformer.h" for GPT-2
LAYER_ATTR = _layer_attr   # used in trace calls below

def _nnsight_layer(model_obj, layer_idx):
    """Return the nnsight-traceable layer proxy for a given index."""
    obj = model_obj
    for part in LAYER_ATTR.split('.'):
        obj = getattr(obj, part)
    return obj[layer_idx]

# ── nnsight helpers ──────────────────────────────────────────
def _resolve(proxy):
    type_name = type(proxy).__name__
    if 'Proxy' in type_name or 'InterventionProxy' in type_name:
        return proxy.value
    if hasattr(proxy, 'value') and not isinstance(proxy, (torch.Tensor, torch.nn.Parameter)):
        return proxy.value
    return proxy

def _get_residual(proxy):
    t = _resolve(proxy)
    return t[0].detach().clone() if t.dim() == 3 else t.detach().clone()

def _get_last_logits(proxy):
    t = _resolve(proxy)
    return t[0, -1, :].float().cpu() if t.dim() == 3 else t[-1, :].float().cpu()

# ── Architecture info (resolved from loaded model) ────────────────────────
inner = model._model
d_model    = getattr(inner.config, 'hidden_size',
             getattr(inner.config, 'n_embd', None))
vocab_size = inner.config.vocab_size
print(f'Architecture: {N_LAYERS} layers, d_model={d_model}, vocab={vocab_size}')

# ── Unembedding matrix ───────────────────────────────────────
def _safe_to_cpu(t):
    if hasattr(t, 'value'): t = t.value
    return t.detach().float().cpu()

def _is_meta(module, attr='weight'):
    try:
        p = getattr(module, attr)
        return p.device.type == 'meta'
    except Exception:
        return True

# Identify lm_head and final norm
lm_head = getattr(inner, 'lm_head', None)
if lm_head is None:
    raise RuntimeError('Cannot find lm_head on model. Add support for this architecture.')

# Some models tie embeddings; force materialization via dummy trace if needed
if _is_meta(lm_head):
    print('Meta tensors detected — running dummy trace to materialize weights...')
    _dummy = tokenizer('The', return_tensors='pt')['input_ids'].to(device)
    with model.trace(_dummy, scan=False, validate=False):
        _ = model.output.logits.save()
    del _dummy
    print('Weights materialized.')

try:
    W_U = lm_head.weight.detach().float().cpu()
    print('W_U from lm_head')
except (NotImplementedError, RuntimeError):
    # Tied weights: use embedding table
    wte = getattr(getattr(inner, 'transformer', inner), 'wte', None) or \
          getattr(getattr(inner, 'model', inner), 'embed_tokens', None)
    if wte is None:
        raise RuntimeError('Cannot locate W_U (lm_head or embed_tokens).')
    W_U = wte.weight.detach().float().cpu()
    print('W_U from tied embedding')
print(f'W_U: {W_U.shape}')

# ── Final LayerNorm ──────────────────────────────────────────
# Try common attribute paths
final_norm_module = None
for path in ('transformer.ln_f', 'model.norm', 'norm'):
    obj = inner
    for part in path.split('.'):
        obj = getattr(obj, part, None)
        if obj is None: break
    if obj is not None and isinstance(obj, torch.nn.Module):
        final_norm_module = obj
        print(f'Final LayerNorm: {path}')
        break
if final_norm_module is None:
    raise RuntimeError('Cannot locate final LayerNorm. Add support for this architecture.')

try:
    _test_ln = final_norm_module(torch.randn(1, d_model).to(device))
    del _test_ln
    print(f'Final LayerNorm verified on {device}')
except Exception as e:
    print(f'  LayerNorm access issue: {e}')

def apply_final_norm(x):
    needs_batch = (x.dim() == 1)
    inp = x.unsqueeze(0) if needs_batch else x
    inp = inp.to(device)
    with torch.no_grad():
        out = final_norm_module(inp).float()
    if needs_batch:
        out = out.squeeze(0)
    return out.cpu()

# ── W_U SVD (for M5 + linear cast) ──────────────────────────
print('Computing W_U SVD...')
W_U_gpu = W_U.float().to(device)
U_wu, S_wu, Vh_wu = torch.linalg.svd(W_U_gpu, full_matrices=False)
N_WU_TOP = 50
wu_top_dirs = Vh_wu[:N_WU_TOP].cpu().float()
S_wu_cpu = S_wu.cpu().float()
total_var = (S_wu_cpu**2).sum().item()
top_var = (S_wu_cpu[:N_WU_TOP]**2).sum().item()
print(f'Top-{N_WU_TOP} SV fraction: {top_var/total_var:.1%}')
del U_wu, W_U_gpu
if device == 'cuda': torch.cuda.empty_cache()

# ── Model dtype ──────────────────────────────────────────────
MODEL_DTYPE = None
for p in inner.parameters():
    if p.device.type != 'meta':
        MODEL_DTYPE = p.dtype
        break
if MODEL_DTYPE is None:
    MODEL_DTYPE = torch.float32
print(f'Model dtype: {MODEL_DTYPE}')

# ── Baseline ─────────────────────────────────────────────────
NEUTRAL_PROMPT = 'In my experience,'
baseline_tokens = tokenizer(NEUTRAL_PROMPT, return_tensors='pt')['input_ids'].to(device)
with model.trace(baseline_tokens, scan=False, validate=False):
    bl_proxy = model.output.logits.save()
baseline_logits = _get_last_logits(bl_proxy)
baseline_probs = F.softmax(baseline_logits, dim=-1)
baseline_ranks = torch.argsort(torch.argsort(baseline_logits, descending=True))
print(f'Baseline: "{NEUTRAL_PROMPT}" -> top token: \'{tokenizer.decode(baseline_logits.argmax().item())}\'')

ALL_LAYERS = list(range(N_LAYERS))
print(f'\nAll infrastructure loaded')


# ── Architecture-agnostic embedding + position embed helpers ─────────────────
# Detect embedding path at model load time (outside trace context)
EMBED_ATTR = None
for _ep in ('model.embed_tokens', 'transformer.wte', 'embed_tokens'):
    _obj = inner
    for _part in _ep.split('.'):
        _obj = getattr(_obj, _part, None)
        if _obj is None: break
    if _obj is not None:
        EMBED_ATTR = _ep
        print(f'Embedding path: {EMBED_ATTR}')
        break
if EMBED_ATTR is None:
    raise RuntimeError('Cannot locate token embedding. Add support for this architecture.')

# Detect position embedding path (None for RoPE models)
POS_EMBED_ATTR = None
for _pp in ('transformer.wpe',):
    _obj = inner
    for _part in _pp.split('.'):
        _obj = getattr(_obj, _part, None)
        if _obj is None: break
    if _obj is not None:
        POS_EMBED_ATTR = _pp
        print(f'Position embed path: {POS_EMBED_ATTR}')
        break
if POS_EMBED_ATTR is None:
    print('Position embed: none (RoPE — position encoding is inside layers)')


def _nnsight_embed(model_obj):
    """Return nnsight-traceable embedding proxy (call inside trace context)."""
    obj = model_obj
    for part in EMBED_ATTR.split('.'):
        obj = getattr(obj, part)
    return obj


def _get_pos_embed(tokens):
    """Return explicit position embedding tensor, or None for RoPE models."""
    if POS_EMBED_ATTR is None:
        return None
    obj = inner
    for part in POS_EMBED_ATTR.split('.'):
        obj = getattr(obj, part)
    seq_len = tokens.shape[1]
    pos_ids = torch.arange(seq_len, device=device).unsqueeze(0)
    with torch.no_grad():
        return obj(pos_ids).float()


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2: PROMPTS
# ════════════════════════════════════════════════════════════════
# CONVERGENCE_PROMPTS, EASY_PROMPTS, MEDIUM_PROMPTS, HARD_PROMPTS
# are already loaded from data/prompts.json by the config cell (Cell 1).
# This cell tokenizes them and defines STRESS_PROMPTS.

convergence_tokens = [
    tokenizer(p, return_tensors='pt')['input_ids'].to(device)
    for p in CONVERGENCE_PROMPTS
]

# A small subset for quick sanity checks (not used in main measurements)
STRESS_PROMPTS = CONVERGENCE_PROMPTS[:3]

print(f'{len(CONVERGENCE_PROMPTS)} convergence + '
      f'{len(EASY_PROMPTS)+len(MEDIUM_PROMPTS)+len(HARD_PROMPTS)} stratified + '
      f'{len(STRESS_PROMPTS)} stress prompts tokenized')


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2B: RUNTIME KNOBS (already resolved from config.yaml)
# ════════════════════════════════════════════════════════════════
# FAST_MODE, M1_N_RAND, M1_N_PROMPTS, M4_N_PROMPTS, PERTURB_SCALE,
# LC_TRAIN_PROMPTS, LC_TEST_PROMPTS are all set by Cell 1 from config.yaml.
# This cell just echoes the resolved values for visibility.

LC_TRAIN_PROMPTS = max(2, len(CONVERGENCE_PROMPTS) - 2)
LC_TEST_PROMPTS  = min(2, max(1, len(CONVERGENCE_PROMPTS) - LC_TRAIN_PROMPTS))

print('Runtime config (from config.yaml):')
print(f'  FAST_MODE              = {FAST_MODE}')
print(f'  CONVERGENCE_PROMPTS    = {len(CONVERGENCE_PROMPTS)}')
print(f'  EASY/MEDIUM/HARD       = {len(EASY_PROMPTS)}/{len(MEDIUM_PROMPTS)}/{len(HARD_PROMPTS)}')
print(f'  M1 dirs={M1_N_RAND}, M1 prompts={M1_N_PROMPTS}, M4 prompts={M4_N_PROMPTS}')
print(f'  Cast train/test        = {LC_TRAIN_PROMPTS}/{LC_TEST_PROMPTS}')
print(f'  PERTURB_SCALE          = {PERTURB_SCALE}')


## Part 4: Boundary Convergence (M1-M6)

Full 6-measurement convergence suite from the boundary replication notebook.

**Runtime:** ~20-30 min (M1 dominates)


In [ ]:
# ════════════════════════════════════════════════════════════════
# PART 4A: M1 — GAIN CROSSOVER (ALL 12 LAYERS)
# ════════════════════════════════════════════════════════════════

tick('part4')
print('='*70)
print('M1: GAIN CROSSOVER')
print('='*70)

N_RAND_M1 = int(globals().get('M1_N_RAND', 24))
N_PROMPTS_M1 = min(int(globals().get('M1_N_PROMPTS', 8)), len(convergence_tokens))

bl_logits_cache = []
for pi in range(N_PROMPTS_M1):
    with model.trace(convergence_tokens[pi], scan=False, validate=False):
        bp = model.output.logits.save()
    bl_logits_cache.append(_get_last_logits(bp))

m1_results = []
for layer_idx in tqdm(ALL_LAYERS, desc='M1 layers'):
    for pi in range(N_PROMPTS_M1):
        tokens = convergence_tokens[pi]
        bl_log = bl_logits_cache[pi]

        with model.trace(tokens, scan=False, validate=False):
            rp = _nnsight_layer(model, layer_idx).output[0].save()
        layer_res = _get_residual(rp)
        x_last = layer_res[-1].float().cpu()

        torch.manual_seed(42 + layer_idx * 1000 + pi)
        dirs = torch.randn(N_RAND_M1, d_model)
        dirs = dirs / dirs.norm(dim=1, keepdim=True)
        deltas = dirs * PERTURB_SCALE

        nc = apply_final_norm(x_last)
        x_pert = x_last.unsqueeze(0) + deltas
        np_all = apply_final_norm(x_pert)
        linear_shifts = (np_all - nc.unsqueeze(0)) @ W_U.T
        linear_norms = linear_shifts.norm(dim=1).numpy()

        actual_norms = []
        layer_res_gpu = layer_res.float().to(device)
        for di in range(N_RAND_M1):
            delta_gpu = deltas[di].to(device)
            patched = layer_res_gpu.clone()
            patched[-1, :] = patched[-1, :] + delta_gpu
            with model.trace(tokens, scan=False, validate=False):
                _nnsight_layer(model, layer_idx).output[0][:] = patched.unsqueeze(0)
                sl = model.output.logits.save()
            steered = _get_last_logits(sl)
            actual_norms.append((steered - bl_log).float().norm().item())

        gains = np.array(actual_norms) / (linear_norms + 1e-10)
        m1_results.append({
            'layer': layer_idx, 'prompt_idx': pi,
            'gain_mean': gains.mean(), 'gain_median': np.median(gains),
            'gain_std': gains.std(), 'frac_above_1': (gains > 1.0).mean(),
        })
    if device == 'cuda': torch.cuda.empty_cache()

m1_df = pd.DataFrame(m1_results)
m1_df.to_csv(f'{PREFIX}_m1_gain_crossover.csv', index=False)

m1_summary = m1_df.groupby('layer').agg({'gain_median': ['mean', 'std'], 'frac_above_1': 'mean'}).reset_index()
m1_summary.columns = ['layer', 'gain_median_mean', 'gain_median_std', 'frac_above_1']

# ── STABLE crossover rule ────────────────────────────────────
# Use sustained threshold crossing, but exclude the terminal layer from the
# "must stay high" condition. The last layer can show a readout pullback.
layers_sorted = m1_summary.sort_values('layer').reset_index(drop=True)
gains_by_layer = layers_sorted['gain_median_mean'].values
frac_by_layer = layers_sorted['frac_above_1'].values
layer_ids = layers_sorted['layer'].astype(int).values

# Exclude final layer from sustained checks.
if len(gains_by_layer) > 1:
    gains_check = gains_by_layer[:-1]
    frac_check = frac_by_layer[:-1]
    layer_check = layer_ids[:-1]
else:
    gains_check = gains_by_layer
    frac_check = frac_by_layer
    layer_check = layer_ids

# Deterministic fallback: layer with max gain (excluding final layer).
fallback_idx = int(np.argmax(gains_check))
crossover_m1 = int(layer_check[fallback_idx])

for start_idx in range(len(gains_check)):
    if np.all(gains_check[start_idx:] >= 1.0):
        crossover_m1 = int(layer_check[start_idx])
        break

# Alternative crossover using fraction-above-1 signal.
crossover_m1_frac = int(layer_check[int(np.argmax(frac_check))])
for start_idx in range(len(frac_check)):
    if np.all(frac_check[start_idx:] >= 0.5):
        crossover_m1_frac = int(layer_check[start_idx])
        break

print(f'  GAIN CROSSOVER (sustained ≥1.0): L{crossover_m1}')
print(f'  GAIN CROSSOVER (frac_above_1 ≥0.5 sustained): L{crossover_m1_frac}')
print(f'  Using sustained-gain rule → L{crossover_m1}')
print(f'  Saved: {PREFIX}_m1_gain_crossover.csv ({len(m1_df)} rows)')


In [ ]:
# ════════════════════════════════════════════════════════════════
# PART 4B: M2 — LOGIT LENS CONVERGENCE
# ════════════════════════════════════════════════════════════════

print('\n' + '='*70)
print('M2: LOGIT LENS CONVERGENCE')
print('='*70)

m2_results = []
for pi, tokens in enumerate(tqdm(convergence_tokens, desc='M2 prompts')):
    saved = {}
    with model.trace(tokens, scan=False, validate=False):
        for l in ALL_LAYERS:
            saved[l] = _nnsight_layer(model, l).output[0].save()
        final_proxy = model.output.logits.save()

    final_logits = _get_last_logits(final_proxy)
    final_lp = F.log_softmax(final_logits, dim=-1)
    final_probs = final_lp.exp()
    final_token = final_logits.argmax().item()

    for l in ALL_LAYERS:
        h = _get_residual(saved[l])[-1].float()
        h_normed = apply_final_norm(h)
        logits_l = h_normed @ W_U.T
        lp_l = F.log_softmax(logits_l, dim=-1)
        probs_l = lp_l.exp()

        kl = F.kl_div(lp_l, final_probs, reduction='sum').item()
        top1 = int(logits_l.argmax().item() == final_token)
        entropy = -(probs_l * lp_l).sum().item()
        rank = (logits_l > logits_l[final_token]).sum().item()

        m2_results.append({
            'layer': l, 'prompt_idx': pi, 'prompt': CONVERGENCE_PROMPTS[pi][:50],
            'kl_from_final': kl, 'top1_match': top1, 'entropy': entropy,
            'correct_token_rank': rank,
        })
    del saved; gc.collect()

m2_df = pd.DataFrame(m2_results)
m2_df.to_csv(f'{PREFIX}_m2_logit_lens.csv', index=False)
m2_agg = m2_df.groupby('layer').agg({'kl_from_final': ['mean','std'], 'top1_match': 'mean', 'entropy': 'mean'}).reset_index()
m2_agg.columns = ['layer', 'kl_mean', 'kl_std', 'top1_acc', 'entropy_mean']
m2_agg.to_csv(f'{PREFIX}_m2_logit_lens_summary.csv', index=False)

kl_max = m2_agg['kl_mean'].max()
converged = m2_agg[m2_agg['kl_mean'] < kl_max * 0.1]['layer']
crossover_m2 = int(converged.min()) if len(converged) > 0 else N_LAYERS // 2
print(f'  KL convergence: L{crossover_m2}')


In [ ]:
# ════════════════════════════════════════════════════════════════
# PART 4C: M3 — REPRESENTATIONAL SIMILARITY
# ════════════════════════════════════════════════════════════════

print('\n' + '='*70)
print('M3: REPRESENTATIONAL SIMILARITY')
print('='*70)

m3_results = []
for pi, tokens in enumerate(tqdm(convergence_tokens, desc='M3 prompts')):
    saved = {}
    with model.trace(tokens, scan=False, validate=False):
        for l in ALL_LAYERS:
            saved[l] = _nnsight_layer(model, l).output[0].save()

    final_vec = _get_residual(saved[N_LAYERS-1])[-1].float()

    for l in ALL_LAYERS:
        h = _get_residual(saved[l])[-1].float()
        cos_final = F.cosine_similarity(h.unsqueeze(0), final_vec.unsqueeze(0)).item()
        norm_ratio = h.norm().item() / (final_vec.norm().item() + 1e-10)
        m3_results.append({'layer': l, 'prompt_idx': pi, 'cos_sim_to_final': cos_final, 'norm_ratio': norm_ratio})
    del saved

m3_df = pd.DataFrame(m3_results)
m3_df.to_csv(f'{PREFIX}_m3_similarity.csv', index=False)
m3_agg = m3_df.groupby('layer').agg({'cos_sim_to_final': ['mean','std'], 'norm_ratio': 'mean'}).reset_index()
m3_agg.columns = ['layer', 'cos_final_mean', 'cos_final_std', 'norm_ratio_mean']
m3_agg.to_csv(f'{PREFIX}_m3_similarity_summary.csv', index=False)

high_sim = m3_agg[m3_agg['cos_final_mean'] > 0.8]['layer']
crossover_m3 = int(high_sim.min()) if len(high_sim) > 0 else N_LAYERS // 2
print(f'  Cos>0.8 onset: L{crossover_m3}')


In [ ]:
# ════════════════════════════════════════════════════════════════
# PART 4D: M4 — LAYER ABLATION
# ════════════════════════════════════════════════════════════════

print('\n' + '='*70)
print('M4: LAYER ABLATION SENSITIVITY')
print('='*70)

m4_results = []
N_PROMPTS_M4 = min(int(globals().get('M4_N_PROMPTS', 8)), len(convergence_tokens))

for pi in tqdm(range(N_PROMPTS_M4), desc='M4 prompts'):
    tokens = convergence_tokens[pi]
    with model.trace(tokens, scan=False, validate=False):
        cp = model.output.logits.save()
    clean_logits = _get_last_logits(cp)
    clean_probs = F.softmax(clean_logits, dim=-1)
    clean_token = clean_logits.argmax().item()

    for l in ALL_LAYERS:
        with model.trace(tokens, scan=False, validate=False):
            if l == 0:
                inp_proxy = _nnsight_embed(model).output.save()
            else:
                inp_proxy = _nnsight_layer(model, l-1).output[0].save()

        inp_val = _resolve(inp_proxy)
        inp_tensor = inp_val.unsqueeze(0) if inp_val.dim() == 2 else inp_val

        if l == 0:
            seq_len = tokens.shape[1]
            pos_ids = torch.arange(seq_len, device=device).unsqueeze(0)
            with torch.no_grad():
                pos_emb = _get_pos_embed(tokens)
            if pos_emb is not None:
                inp_tensor = (inp_tensor.float() + pos_emb.float()).to(inp_tensor.dtype)

        with model.trace(tokens, scan=False, validate=False):
            _nnsight_layer(model, l).output[0][:] = inp_tensor
            ap = model.output.logits.save()
        abl_logits = _get_last_logits(ap)

        kl = F.kl_div(F.log_softmax(abl_logits, dim=-1), clean_probs, reduction='sum').item()
        shift = (abl_logits - clean_logits).norm().item()
        m4_results.append({'layer': l, 'prompt_idx': pi, 'ablation_kl': kl, 'logit_shift': shift})

m4_df = pd.DataFrame(m4_results)
m4_df.to_csv(f'{PREFIX}_m4_ablation.csv', index=False)
m4_agg = m4_df.groupby('layer').agg({'ablation_kl': ['mean','std'], 'logit_shift': 'mean'}).reset_index()
m4_agg.columns = ['layer', 'abl_kl_mean', 'abl_kl_std', 'logit_shift_mean']
m4_agg.to_csv(f'{PREFIX}_m4_ablation_summary.csv', index=False)

abl_med = m4_agg['abl_kl_mean'].median()
late_low = m4_agg[(m4_agg['abl_kl_mean'] < abl_med) & (m4_agg['layer'] > N_LAYERS//4)]
crossover_m4 = int(late_low['layer'].min()) if len(late_low) > 0 else N_LAYERS // 2
print(f'  Redundancy onset: L{crossover_m4}')


In [ ]:
# ════════════════════════════════════════════════════════════════
# PART 4E: M5 — HOURGLASS (UPDATE ALIGNMENT)
# ════════════════════════════════════════════════════════════════

print('\n' + '='*70)
print('M5: UPDATE ALIGNMENT (HOURGLASS)')
print('='*70)

m5_results = []
for pi, tokens in enumerate(tqdm(convergence_tokens, desc='M5 prompts')):
    saved = {}
    with model.trace(tokens, scan=False, validate=False):
        for l in ALL_LAYERS:
            saved[l] = _nnsight_layer(model, l).output[0].save()

    for l in ALL_LAYERS:
        h_out = _get_residual(saved[l])[-1].float()
        if l == 0:
            m5_results.append({'layer': l, 'prompt_idx': pi, 'output_alignment': float('nan'), 'update_norm': float('nan')})
            continue
        h_in = _get_residual(saved[l-1])[-1].float()
        update = h_out - h_in
        un = update.norm().item()
        if un < 1e-10:
            m5_results.append({'layer': l, 'prompt_idx': pi, 'output_alignment': 0.0, 'update_norm': 0.0})
            continue
        update_unit = update / un
        projections = wu_top_dirs @ update_unit.cpu()
        output_alignment = (projections ** 2).sum().item()
        m5_results.append({'layer': l, 'prompt_idx': pi, 'output_alignment': output_alignment, 'update_norm': un})
    del saved

m5_df = pd.DataFrame(m5_results)
m5_df.to_csv(f'{PREFIX}_m5_alignment.csv', index=False)
m5_agg = m5_df.dropna().groupby('layer').agg({'output_alignment': ['mean','std'], 'update_norm': 'mean'}).reset_index()
m5_agg.columns = ['layer', 'align_mean', 'align_std', 'update_norm_mean']
m5_agg.to_csv(f'{PREFIX}_m5_alignment_summary.csv', index=False)

early_align = m5_agg[m5_agg['layer'] <= 3]['align_mean'].mean()
rising = m5_agg[m5_agg['align_mean'] > early_align * 2.0]['layer']
crossover_m5 = int(rising.min()) if len(rising) > 0 else N_LAYERS // 2
print(f'  Output alignment onset: L{crossover_m5}')


In [ ]:
# ════════════════════════════════════════════════════════════════
# PART 4F: M6 — CERTAINTY-DRIVEN CONVERGENCE
# ════════════════════════════════════════════════════════════════

print('\n' + '='*70)
print('M6: CERTAINTY-DRIVEN CONVERGENCE')
print('='*70)

m6_results = []
for difficulty, prompts in [('easy', EASY_PROMPTS), ('medium', MEDIUM_PROMPTS), ('hard', HARD_PROMPTS)]:
    for prompt in tqdm(prompts, desc=f'M6 {difficulty}'):
        tokens = tokenizer(prompt, return_tensors='pt')['input_ids'].to(device)
        saved = {}
        with model.trace(tokens, scan=False, validate=False):
            for l in ALL_LAYERS:
                saved[l] = _nnsight_layer(model, l).output[0].save()
            fp = model.output.logits.save()

        fl = _get_last_logits(fp)
        flp = F.log_softmax(fl, dim=-1)
        fp_ = flp.exp()
        fe = -(fp_ * flp).sum().item()

        for l in ALL_LAYERS:
            h = _get_residual(saved[l])[-1].float()
            hn = apply_final_norm(h)
            ll = hn @ W_U.T
            lp = F.log_softmax(ll, dim=-1)
            kl = F.kl_div(lp, fp_, reduction='sum').item()
            top1 = int(ll.argmax().item() == fl.argmax().item())
            m6_results.append({'layer': l, 'difficulty': difficulty, 'prompt': prompt[:50],
                               'kl_from_final': kl, 'top1_match': top1, 'final_entropy': fe})
        del saved

m6_df = pd.DataFrame(m6_results)
m6_df.to_csv(f'{PREFIX}_m6_certainty.csv', index=False)
m6_agg = m6_df.groupby(['layer','difficulty']).agg({'kl_from_final': 'mean', 'final_entropy': 'mean'}).reset_index()
m6_agg.to_csv(f'{PREFIX}_m6_certainty_summary.csv', index=False)

# ── Derivative-based knee detection ──────────────────────────
# Previous method: threshold (kl < kl_peak * 0.1) → collapsed to L10-11
# New method: max curvature of log(KL) on L0-10, same as find_knee in Part 5B
# For per-difficulty headline: compute knees per-prompt first, then median

def find_convergence_knee(kl_values, layers):
    """Find convergence knee via max curvature of log(KL).
    Excludes final layer (trivially ≈ 0). Handles non-monotone early
    bumps by searching from the post-peak region.
    """
    kl_arr = np.array(kl_values, dtype=float)
    layers_arr = np.array(layers, dtype=int)
    # Exclude final layer (trivially near zero)
    if len(kl_arr) > 3:
        kl_use = kl_arr[:-1]
        layers_use = layers_arr[:-1]
    else:
        kl_use = kl_arr
        layers_use = layers_arr
    log_kl = np.log(np.clip(kl_use, 1e-10, None))
    if len(log_kl) < 3:
        return int(layers_use[len(layers_use)//2])

    # Find the peak of log_kl, then only look at post-peak region for knee
    # This avoids nonsense knees from early non-monotone bumps
    peak_idx = int(np.argmax(log_kl))
    # Need at least 3 points post-peak for second derivative
    if len(log_kl) - peak_idx >= 3:
        log_kl_post = log_kl[peak_idx:]
        layers_post = layers_use[peak_idx:]
    else:
        # Not enough post-peak data, use full range
        log_kl_post = log_kl
        layers_post = layers_use

    d2 = np.diff(log_kl_post, 2)
    knee_idx = int(np.argmin(d2)) + 1  # +1 because diff loses first element
    return int(layers_post[knee_idx])

# ── Per-prompt convergence (knee method) ─────────────────────
per_prompt = []
for (prompt, diff), sub in m6_df.groupby(['prompt','difficulty']):
    sub = sub.sort_values('layer')
    cl = find_convergence_knee(sub['kl_from_final'].values, sub['layer'].values)
    per_prompt.append({'prompt': prompt, 'difficulty': diff,
                       'convergence_layer': cl, 'final_entropy': sub['final_entropy'].iloc[0]})
ppc_df = pd.DataFrame(per_prompt)
ppc_df.to_csv(f'{PREFIX}_m6_per_prompt.csv', index=False)

# ── Per-difficulty headline: median of per-prompt knees ──────
# (More robust than running knee on mean curve, which can create
#  nonsense like "medium knee L1" from early bumps in the average)
m6_crossovers = {}
for diff in ['easy', 'medium', 'hard']:
    diff_knees = ppc_df[ppc_df['difficulty']==diff]['convergence_layer']
    median_knee = int(np.median(diff_knees))
    m6_crossovers[diff] = median_knee
    print(f'  {diff}: median knee L{median_knee} (per-prompt: {diff_knees.tolist()})')

r_dyn, p_dyn = stats.spearmanr(ppc_df['final_entropy'], ppc_df['convergence_layer'])
shift = m6_crossovers.get('hard',0) - m6_crossovers.get('easy',0)
print(f'  ρ(entropy, convergence_layer) = {r_dyn:+.3f} (p={p_dyn:.2e})')
print(f'  Boundary shift (hard-easy): {shift:+d} layers')


In [ ]:
# ════════════════════════════════════════════════════════════════
# PART 4G: CONVERGENCE FIGURE
# ════════════════════════════════════════════════════════════════

# ── Build crossover table with CORRECTED depth fractions ──────
# Fix: layer/(N_LAYERS-1) so L11 → 100% (not layer/N_LAYERS which gives 91.7%)
# Also uses the stable M1 crossover and raw knee for M2

# ── Compute M2 raw knee here (from m2_agg, already available from Part 4B) ──
# Same algorithm as find_knee in Part 5B: second derivative of log(KL), L0-10
def _find_knee_from_agg(kl_values, layers):
    """Knee detection on log(KL), excluding final layer."""
    kl_arr = np.array(kl_values, dtype=float)
    layers_arr = np.array(layers, dtype=int)
    if len(kl_arr) > 3:
        kl_use = kl_arr[:-1]
        layers_use = layers_arr[:-1]
    else:
        kl_use = kl_arr
        layers_use = layers_arr
    log_kl = np.log(np.clip(kl_use, 1e-10, None))
    if len(log_kl) < 3:
        return int(layers_use[len(layers_use)//2])
    # Post-peak search to avoid early bumps
    peak_idx = int(np.argmax(log_kl))
    if len(log_kl) - peak_idx >= 3:
        log_kl_post = log_kl[peak_idx:]
        layers_post = layers_use[peak_idx:]
    else:
        log_kl_post = log_kl
        layers_post = layers_use
    d2 = np.diff(log_kl_post, 2)
    knee_idx = int(np.argmin(d2)) + 1
    return int(layers_post[knee_idx])

m2_sorted = m2_agg.sort_values('layer')
knee_raw = _find_knee_from_agg(m2_sorted['kl_mean'].values, m2_sorted['layer'].values)
print(f'  M2 raw knee (log KL, L0-{N_LAYERS-2}): L{knee_raw}')

crossovers = {
    'M1: Gain': crossover_m1, 'M2: Logit lens (raw knee)': knee_raw,
    'M3: Cos sim': crossover_m3, 'M4: Ablation': crossover_m4,
    'M5: Hourglass': crossover_m5,
}
cross_vals = list(crossovers.values())
cross_mean = np.mean(cross_vals)
cross_range = max(cross_vals) - min(cross_vals)

print(f'\nTransition estimates (depth = layer/(L-1)):')
for name, val in crossovers.items():
    print(f'  {name:30s}  L{val} ({val/(N_LAYERS-1)*100:.1f}%)')
print(f'Mean: L{cross_mean:.1f} ({cross_mean/(N_LAYERS-1)*100:.1f}%)')
# Note: cast knee will be computed and reported in Part 5B

cross_df = pd.DataFrame([{'measurement': k, 'layer': v,
                           'pct_depth': v/(N_LAYERS-1)*100} for k, v in crossovers.items()])
cross_df.to_csv(f'{PREFIX}_convergence_crossovers.csv', index=False)

def norm01(v):
    v = np.array(v, dtype=float)
    return (v - v.min()) / (v.max() - v.min() + 1e-10)

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
fig.suptitle(f'GPT-2 Encoder-Decoder Boundary ({N_LAYERS} layers)\n'
             f'Depth = layer/(L-1) | M1 crossover = sustained ≥1.0 rule | M2 = raw knee on log(KL)',
             fontsize=13, fontweight='bold')
bc = 'red'

ax = axes[0,0]
g = m1_df.groupby('layer')['gain_median'].mean()
gs = m1_df.groupby('layer')['gain_median'].std()
ax.plot(g.index, g.values, 'b-o', ms=5)
ax.fill_between(g.index, g.values-gs.values, g.values+gs.values, alpha=0.2)
ax.axhline(1.0, color='gray', ls='--'); ax.axvline(crossover_m1, color=bc, ls=':')
ax.set_title(f'M1: Gain (sustained ≥1.0 → L{crossover_m1})'); ax.set_ylabel('Gain')

ax = axes[0,1]
ax.plot(m2_agg['layer'], m2_agg['kl_mean'], 'g-o', ms=5)
ax.fill_between(m2_agg['layer'], m2_agg['kl_mean']-m2_agg['kl_std'], m2_agg['kl_mean']+m2_agg['kl_std'], alpha=0.2, color='green')
ax.axvline(knee_raw, color=bc, ls=':')
ax.set_title(f'M2: Logit Lens (raw knee L{knee_raw})'); ax.set_ylabel('KL')

ax = axes[1,0]
ax.plot(m3_agg['layer'], m3_agg['cos_final_mean'], 'm-o', ms=5)
ax.fill_between(m3_agg['layer'], m3_agg['cos_final_mean']-m3_agg['cos_final_std'], m3_agg['cos_final_mean']+m3_agg['cos_final_std'], alpha=0.2, color='purple')
ax.axhline(0.8, color='gray', ls='--'); ax.axvline(crossover_m3, color=bc, ls=':')
ax.set_title(f'M3: Similarity (L{crossover_m3})'); ax.set_ylabel('Cos Sim')

ax = axes[1,1]
ax.plot(m4_agg['layer'], m4_agg['abl_kl_mean'], 'c-o', ms=5)
ax.fill_between(m4_agg['layer'], m4_agg['abl_kl_mean']-m4_agg['abl_kl_std'], m4_agg['abl_kl_mean']+m4_agg['abl_kl_std'], alpha=0.2, color='cyan')
ax.axvline(crossover_m4, color=bc, ls=':'); ax.set_title(f'M4: Ablation (L{crossover_m4})'); ax.set_ylabel('Ablation KL')

ax = axes[2,0]
ax.plot(m5_agg['layer'], m5_agg['align_mean'], color='orange', marker='o', ms=5)
ax.fill_between(m5_agg['layer'], m5_agg['align_mean']-m5_agg['align_std'], m5_agg['align_mean']+m5_agg['align_std'], alpha=0.2, color='orange')
ax.axvline(crossover_m5, color=bc, ls=':'); ax.set_title(f'M5: Hourglass (L{crossover_m5})'); ax.set_ylabel('W_U Alignment')

ax = axes[2,1]
ax.plot(ALL_LAYERS, norm01(m1_df.groupby('layer')['gain_median'].mean().values), label='M1', lw=2, color='blue')
ax.plot(ALL_LAYERS, 1-norm01(m2_agg['kl_mean'].values), label='M2: 1-KL', lw=2, color='green')
ax.plot(ALL_LAYERS, m3_agg['cos_final_mean'].values, label='M3', lw=2, color='purple')
ax.plot(ALL_LAYERS, 1-norm01(m4_agg['abl_kl_mean'].values), label='M4: 1-AblKL', lw=2, color='cyan')
al = m5_agg['align_mean'].values
al_pad = np.concatenate([[al[0]], al]) if len(al) < N_LAYERS else al
ax.plot(list(range(len(al_pad))), norm01(al_pad), label='M5', lw=2, color='orange')
ax.axvspan(min(cross_vals), max(cross_vals), alpha=0.1, color='red')
ax.axvline(cross_mean, color='red', ls='-', lw=2, alpha=0.8, label=f'Mean L{cross_mean:.0f}')
ax.set_title('ALL MEASUREMENTS (corrected)'); ax.legend(fontsize=7); ax.set_ylim(-0.05, 1.05)

for row in axes:
    for ax in row:
        ax.set_xlabel('Layer')

plt.tight_layout()
plt.savefig(f'{PREFIX}_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

# M6 figure — using median-of-knees per difficulty
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('M6: Certainty-Driven Convergence — GPT-2 (derivative knee, median-of-prompts)', fontsize=13, fontweight='bold')
dc = {'easy': 'green', 'medium': 'orange', 'hard': 'red'}

ax = axes[0]
for diff in ['easy', 'medium', 'hard']:
    sub = m6_agg[m6_agg['difficulty']==diff].sort_values('layer')
    ax.plot(sub['layer'], sub['kl_from_final'], color=dc[diff],
            label=f'{diff} (median knee → L{m6_crossovers[diff]})', lw=2)
ax.set_xlabel('Layer'); ax.set_ylabel('KL'); ax.set_title('KL by Difficulty (knee = median of per-prompt knees)'); ax.legend()

ax = axes[1]
for diff in ['easy', 'medium', 'hard']:
    sub = ppc_df[ppc_df['difficulty']==diff]
    ax.scatter(sub['final_entropy'], sub['convergence_layer'], c=dc[diff], label=diff, s=80, edgecolors='k', lw=0.5)
ax.set_xlabel('Final Entropy'); ax.set_ylabel('Convergence Layer')
ax.set_title(f'ρ={r_dyn:+.3f} (p={p_dyn:.2e})'); ax.legend()

plt.tight_layout()
plt.savefig(f'{PREFIX}_m6_certainty.png', dpi=150, bbox_inches='tight')
plt.show()
tock('part4')


## Part 5: Linear Cast (Option A — Tuned Lens Lite)

Tests whether phase markers (M2 knee, gain crossover) are artifacts of reading out with the wrong basis.

**Method:** Ridge regression per layer: h_ℓ → h_final. Then recompute M2 and gain using the casted readout.

**If the knee disappears:** your raw M2 was mostly "wrong basis."
**If the knee persists:** an actual computation phase transition.

**Runtime:** ~10-15 min (data collection + ridge fit + casted metrics)


In [ ]:
# ════════════════════════════════════════════════════════════════
# PART 5A: LINEAR CAST — TRAIN RIDGE REGRESSION PER LAYER
# ════════════════════════════════════════════════════════════════
# For each layer ℓ, fit: h_final_hat = h_ℓ @ M_ℓ + b_ℓ
# Using WikiText-style data (our convergence prompts as training set,
# plus a held-out set for evaluation).

tick('part5')
print('='*70)
print('PART 5: LINEAR CAST (TUNED LENS LITE)')
print('='*70)

from sklearn.linear_model import Ridge

# ── Step 1: Collect (h_ℓ, h_final) training pairs ───────────
LC_TRAIN_PROMPTS = int(globals().get('LC_TRAIN_PROMPTS', 10))
LC_TEST_PROMPTS = int(globals().get('LC_TEST_PROMPTS', 2))
LC_TRAIN_PROMPTS = min(LC_TRAIN_PROMPTS, max(2, len(CONVERGENCE_PROMPTS)-1))
LC_TEST_PROMPTS = min(LC_TEST_PROMPTS, max(1, len(CONVERGENCE_PROMPTS)-LC_TRAIN_PROMPTS))
TRAIN_PROMPTS = CONVERGENCE_PROMPTS[:LC_TRAIN_PROMPTS]
TEST_PROMPTS_LC = CONVERGENCE_PROMPTS[LC_TRAIN_PROMPTS:LC_TRAIN_PROMPTS+LC_TEST_PROMPTS]

print(f'  Training on {len(TRAIN_PROMPTS)} prompts, testing on {len(TEST_PROMPTS_LC)}')

def collect_hidden_pairs(prompts):
    """Collect (h_ℓ, h_final) pairs for all layers from a set of prompts.
    Returns dict: layer → (X_ℓ, Y) where X_ℓ is (n_tokens, d_model) and Y is (n_tokens, d_model).
    """
    layer_xs = {l: [] for l in ALL_LAYERS}
    ys = []

    for prompt in tqdm(prompts, desc='  collecting hiddens'):
        tokens = tokenizer(prompt, return_tensors='pt')['input_ids'].to(device)
        saved = {}
        with model.trace(tokens, scan=False, validate=False):
            for l in ALL_LAYERS:
                saved[l] = _nnsight_layer(model, l).output[0].save()

        # Final layer hidden = after last transformer block
        h_final = _get_residual(saved[N_LAYERS-1]).float().cpu()
        ys.append(h_final)  # (seq_len, d_model)

        for l in ALL_LAYERS:
            h_l = _get_residual(saved[l]).float().cpu()
            layer_xs[l].append(h_l)

        del saved
        if device == 'cuda': torch.cuda.empty_cache()

    # Concatenate across all prompts (tokens)
    Y = torch.cat(ys, dim=0).numpy()  # (total_tokens, d_model)
    X = {l: torch.cat(layer_xs[l], dim=0).numpy() for l in ALL_LAYERS}
    return X, Y

print('\nCollecting training hidden states...')
X_train, Y_train = collect_hidden_pairs(TRAIN_PROMPTS)
print(f'  Training data: {Y_train.shape[0]} tokens × {Y_train.shape[1]} dims')

print('Collecting test hidden states...')
X_test, Y_test = collect_hidden_pairs(TEST_PROMPTS_LC)
print(f'  Test data: {Y_test.shape[0]} tokens × {Y_test.shape[1]} dims')

# ── Step 2: Fit ridge regression per layer ───────────────────
RIDGE_ALPHA = 1.0  # regularization strength

cast_models = {}  # layer → (M, b)
cast_fit_results = []

print(f'\nFitting ridge regression (α={RIDGE_ALPHA})...')
for l in tqdm(ALL_LAYERS, desc='  fitting'):
    reg = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True)
    reg.fit(X_train[l], Y_train)

    # Train R²
    train_r2 = reg.score(X_train[l], Y_train)
    # Test R²
    test_r2 = reg.score(X_test[l], Y_test)

    # Store as torch tensors for inference
    M = torch.from_numpy(reg.coef_.T).float()  # (d_model, d_model)
    b = torch.from_numpy(reg.intercept_).float()  # (d_model,)
    cast_models[l] = (M, b)

    # Cosine similarity of predictions to actual (test set)
    Y_pred = X_test[l] @ reg.coef_.T + reg.intercept_
    cos_sims = []
    for i in range(len(Y_test)):
        cs = np.dot(Y_pred[i], Y_test[i]) / (np.linalg.norm(Y_pred[i]) * np.linalg.norm(Y_test[i]) + 1e-10)
        cos_sims.append(cs)

    cast_fit_results.append({
        'layer': l, 'train_r2': train_r2, 'test_r2': test_r2,
        'mean_cos_sim': np.mean(cos_sims), 'std_cos_sim': np.std(cos_sims),
    })
    print(f'  L{l:2d}: train R²={train_r2:.4f}, test R²={test_r2:.4f}, cos={np.mean(cos_sims):.4f}')

cast_fit_df = pd.DataFrame(cast_fit_results)
cast_fit_df.to_csv(f'{PREFIX}_linear_cast_fit.csv', index=False)
print(f'\nSaved: {PREFIX}_linear_cast_fit.csv')


In [ ]:
# ════════════════════════════════════════════════════════════════
# PART 5B: CASTED LOGIT LENS + KNEE DETECTION
# ════════════════════════════════════════════════════════════════
# Replace raw logit lens readout with casted readout.
# If M2 knee disappears → old marker was basis mismatch.

print('\n' + '='*70)
print('CASTED LOGIT LENS')
print('='*70)

def casted_logit_lens(h_l, layer_idx):
    """Apply linear cast then logit lens.
    h_final_hat = h_l @ M + b
    logits = apply_final_norm(h_final_hat) @ W_U^T
    """
    M, b = cast_models[layer_idx]
    h_cast = h_l.float().cpu() @ M + b
    h_normed = apply_final_norm(h_cast)
    logits = h_normed @ W_U.T
    return logits

casted_m2_results = []
raw_m2_results = []  # recompute raw for direct comparison on same prompts

for pi, tokens in enumerate(tqdm(convergence_tokens, desc='Casted M2')):
    saved = {}
    with model.trace(tokens, scan=False, validate=False):
        for l in ALL_LAYERS:
            saved[l] = _nnsight_layer(model, l).output[0].save()
        fp = model.output.logits.save()

    final_logits = _get_last_logits(fp)
    final_lp = F.log_softmax(final_logits, dim=-1)
    final_probs = final_lp.exp()
    final_token = final_logits.argmax().item()

    for l in ALL_LAYERS:
        h = _get_residual(saved[l])[-1].float()

        # ── Raw logit lens ──
        h_normed_raw = apply_final_norm(h)
        logits_raw = h_normed_raw @ W_U.T
        lp_raw = F.log_softmax(logits_raw, dim=-1)
        kl_raw = F.kl_div(lp_raw, final_probs, reduction='sum').item()
        top1_raw = int(logits_raw.argmax().item() == final_token)

        raw_m2_results.append({
            'layer': l, 'prompt_idx': pi, 'kl_from_final': kl_raw,
            'top1_match': top1_raw, 'lens': 'raw',
        })

        # ── Casted logit lens ──
        logits_cast = casted_logit_lens(h, l)
        lp_cast = F.log_softmax(logits_cast, dim=-1)
        kl_cast = F.kl_div(lp_cast, final_probs, reduction='sum').item()
        top1_cast = int(logits_cast.argmax().item() == final_token)

        casted_m2_results.append({
            'layer': l, 'prompt_idx': pi, 'kl_from_final': kl_cast,
            'top1_match': top1_cast, 'lens': 'casted',
        })

    del saved

# Combine
all_lens_results = raw_m2_results + casted_m2_results
lens_df = pd.DataFrame(all_lens_results)
lens_df.to_csv(f'{PREFIX}_casted_logit_lens.csv', index=False)

# Aggregate
lens_agg = lens_df.groupby(['layer', 'lens']).agg({
    'kl_from_final': ['mean', 'std'], 'top1_match': 'mean',
}).reset_index()
lens_agg.columns = ['layer', 'lens', 'kl_mean', 'kl_std', 'top1_acc']
lens_agg.to_csv(f'{PREFIX}_casted_logit_lens_summary.csv', index=False)

# ── Knee detection ───────────────────────────────────────────
# Knee = layer where log(KL) drops sharply (largest second derivative of log(KL))

def find_knee(kl_values, layers):
    """Find knee via max curvature of log(KL), excluding final layer.
    Uses second derivative of log(KL) — the elbow where convergence
    rate changes most sharply. No arbitrary thresholds.
    """
    kl_arr = np.array(kl_values)
    layers_arr = np.array(layers)
    # Exclude final layer (trivially near zero)
    if len(kl_arr) > 3:
        kl_use = kl_arr[:-1]
        layers_use = layers_arr[:-1]
    else:
        kl_use = kl_arr
        layers_use = layers_arr
    log_kl = np.log(kl_use + 1e-10)
    if len(log_kl) < 3:
        return layers_use[len(layers_use)//2]
    # Second derivative of log(KL) — most negative = steepest acceleration of convergence
    d2 = np.diff(log_kl, 2)
    knee_idx = int(np.argmin(d2)) + 1  # +1 because diff loses first element
    return layers_use[knee_idx]

raw_agg = lens_agg[lens_agg['lens'] == 'raw'].sort_values('layer')
cast_agg = lens_agg[lens_agg['lens'] == 'casted'].sort_values('layer')

knee_raw = find_knee(raw_agg['kl_mean'].values, raw_agg['layer'].values)
knee_cast = find_knee(cast_agg['kl_mean'].values, cast_agg['layer'].values)

print(f'\n  RAW logit lens knee:    L{knee_raw}')
print(f'  CASTED logit lens knee: L{knee_cast}')
print(f'  Difference: {knee_cast - knee_raw:+d} layers')

if abs(knee_cast - knee_raw) <= 1:
    print(f'  → SAME knee location: phase transition is REAL, not lens mismatch')
elif knee_cast < knee_raw:
    print(f'  → Knee shifts EARLIER: raw M2 was partly basis mismatch')
else:
    print(f'  → Knee shifts LATER: casted lens converges slower (unexpected)')


In [ ]:
# ????????????????????????????????????????????????????????????????
# FINAL: THESIS CONFIRMATION CHECKS
# ????????????????????????????????????????????????????????????????

print('='*70)
print('GPT-2 THESIS CONFIRMATION CHECKS')
print('='*70)

# Raw vs casted commitment profile
raw = m2_agg.sort_values('layer').reset_index(drop=True)
lens_agg_local = lens_agg if 'lens_agg' in globals() else pd.read_csv(f'{PREFIX}_casted_logit_lens_summary.csv')
raw_lens = lens_agg_local[lens_agg_local['lens']=='raw'].sort_values('layer').reset_index(drop=True)
cast_lens = lens_agg_local[lens_agg_local['lens']=='casted'].sort_values('layer').reset_index(drop=True)

early_layers = list(range(max(1, N_LAYERS//3)))
late_layers = list(range(N_LAYERS - max(3, N_LAYERS//3), N_LAYERS))

raw_early = raw_lens[raw_lens['layer'].isin(early_layers)]['top1_acc'].mean()
raw_late = raw_lens[raw_lens['layer'].isin(late_layers)]['top1_acc'].mean()
cast_early = cast_lens[cast_lens['layer'].isin(early_layers)]['top1_acc'].mean()

# Hourglass and late-ramp signals
abl = m4_agg.sort_values('layer').reset_index(drop=True)
edge_rows_with_l0 = pd.concat([abl.head(2), abl.tail(2)])
edge_rows_no_l0 = pd.concat([abl.iloc[[1]], abl.tail(2)]) if len(abl) >= 3 else edge_rows_with_l0
edge_mean = edge_rows_no_l0['abl_kl_mean'].mean()
edge_mean_with_l0 = edge_rows_with_l0['abl_kl_mean'].mean()
mid_band = abl[(abl['layer'] >= N_LAYERS//3) & (abl['layer'] < 2*N_LAYERS//3)]
mid_mean = mid_band['abl_kl_mean'].mean()
hourglass_ratio = float(edge_mean / (mid_mean + 1e-10))
hourglass_ratio_with_l0 = float(edge_mean_with_l0 / (mid_mean + 1e-10))

align = m5_agg.sort_values('layer').reset_index(drop=True)
late_update = align[align['layer'].isin(late_layers)]['update_norm_mean'].mean()
mid_update = align[(align['layer'] >= N_LAYERS//3) & (align['layer'] < 2*N_LAYERS//3)]['update_norm_mean'].mean()
late_ramp_ratio = late_update / (mid_update + 1e-10)

checks = [
    ('Raw commitment is late', raw_late >= 0.85 and (raw_late - raw_early) >= 0.50, f'raw early={raw_early:.3f}, raw late={raw_late:.3f}'),
    ('Casted lens reveals early structure', (cast_early - raw_early) >= 0.35, f'cast early={cast_early:.3f}, raw early={raw_early:.3f}'),
    ('M3 high-similarity onset is late', crossover_m3 >= int(0.60*(N_LAYERS-1)), f'crossover_m3=L{crossover_m3}'),
    ('Hourglass sensitivity present', hourglass_ratio >= 2.0, f'edge/mid ratio(noL0)={hourglass_ratio:.2f}, withL0={hourglass_ratio_with_l0:.2f}'),
    ('Late update ramp present', late_ramp_ratio >= 1.5, f'late/mid update ratio={late_ramp_ratio:.2f}'),
]

rows=[]
for name, ok, detail in checks:
    status='PASS' if ok else 'FAIL'
    print(f'  {status:4s} | {name:38s} | {detail}')
    rows.append({'check':name,'status':status,'detail':detail})

checks_df = pd.DataFrame(rows)
checks_df.to_csv(f'{PREFIX}_confirmation_checks.csv', index=False)

overall = 'PASS' if (checks_df['status']=='PASS').all() else 'PARTIAL'
summary = {
    'prefix': PREFIX,
    'overall': overall,
    'm1_crossover': int(crossover_m1),
    'm2_raw_knee': int(knee_raw) if 'knee_raw' in globals() else None,
    'm2_cast_knee': int(knee_cast) if 'knee_cast' in globals() else None,
    'm3_crossover': int(crossover_m3),
    'm4_crossover': int(crossover_m4),
    'm5_crossover': int(crossover_m5),
    'raw_early_top1': float(raw_early),
    'raw_late_top1': float(raw_late),
    'cast_early_top1': float(cast_early),
    'hourglass_ratio': float(hourglass_ratio),
    'hourglass_ratio_no_l0': float(hourglass_ratio),
    'hourglass_ratio_with_l0': float(hourglass_ratio_with_l0),
    'late_ramp_ratio': float(late_ramp_ratio),
}
pd.Series(summary).to_json(f'{PREFIX}_confirmation_summary.json', indent=2)

print('\nOverall:', overall)
print(f'Saved: {PREFIX}_confirmation_checks.csv, {PREFIX}_confirmation_summary.json')


